# 03 — Reddit Post Collection

Collects recent posts from moderation-heavy subreddits using PRAW.
Removal status is inferred from `selftext == '[removed]'`.

**Before running:** add your Reddit API credentials to `.env`:
```
REDDIT_CLIENT_ID=your_id
REDDIT_CLIENT_SECRET=your_secret
REDDIT_USER_AGENT=dsci511-moderation-research/1.0
```
Get credentials at https://www.reddit.com/prefs/apps (choose *script* app type).

Data stored in `../data/moderation.db` → table `reddit_posts`.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
from src.reddit_scraper import get_conn, setup_db, SUBREDDITS

print("Target subreddits:", SUBREDDITS)

## 1. Credential check

In [ ]:
from src.reddit_scraper import get_reddit_client

reddit = get_reddit_client()
# Lightweight check: fetch one post from a known subreddit
test_post = next(reddit.subreddit("announcements").new(limit=1))
print(f"✓ PRAW connected — sample title: {test_post.title[:80]}")

## 2. Database setup

In [ ]:
setup_db()
print("✓ Schema ready")

## 3. Collect posts

In [ ]:
from src.reddit_scraper import collect_all_subreddits

total = collect_all_subreddits()
print(f"\nTotal new posts stored: {total}")

## 4. Row counts

In [ ]:
conn = get_conn()
n_total   = conn.execute("SELECT COUNT(*) FROM reddit_posts").fetchone()[0]
n_removed = conn.execute("SELECT COUNT(*) FROM reddit_posts WHERE is_removed=1").fetchone()[0]
conn.close()

print(f"  reddit_posts total   : {n_total}")
print(f"  reddit_posts removed : {n_removed} ({n_removed/n_total*100:.1f}% removal rate)" if n_total else "  No posts yet")

## 5. Sample records

In [ ]:
conn = get_conn()
df = pd.read_sql_query("""
    SELECT
        subreddit,
        SUBSTR(title, 1, 80) AS title_preview,
        is_removed,
        score,
        num_comments,
        post_created_at
    FROM reddit_posts
    ORDER BY is_removed DESC, post_created_at DESC
    LIMIT 10
""", conn)
conn.close()

pd.set_option("display.max_colwidth", 85)
df

## 6. Removal rate by subreddit

In [ ]:
conn = get_conn()
df_sub = pd.read_sql_query("""
    SELECT
        subreddit,
        COUNT(*) AS total_posts,
        SUM(is_removed) AS removed_posts,
        ROUND(100.0 * SUM(is_removed) / COUNT(*), 1) AS removal_pct
    FROM reddit_posts
    GROUP BY subreddit
    ORDER BY removal_pct DESC
""", conn)
conn.close()
print(df_sub.to_string(index=False))